# Baseline — Direct LLM 烟雾测试版

直接调 SiliconFlow LLM（用 `config.SILICONFLOW_*`），对每个根评论做单步 BULLISH / BEARISH 分类，作为后续 Debate + Judge 流程的对照基线。

默认 `SMOKE=True`，N = 5；需要 `.env` 里存在 `SILICONFLOW_API_KEY`。响应会按 payload SHA256 缓存到 `outputs/llm_cache/direct_llm/`，二次运行只读取缓存。

约束与 `CLAUDE.md` 对齐：**不传 `p1`、未来价格、真实标签、`delta_p`**；只用 `post_content + root_comment.text + replies`，并显式声明 `t_window` 让模型知道要预测的时长。

In [ ]:
import hashlib
import http.client
import json
import re
import socket
import ssl
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (  # noqa: E402
    DEFAULT_INPUT_PATH,
    SILICONFLOW_API_KEY,
    SILICONFLOW_CACHE_DIR,
    SILICONFLOW_ENABLE_THINKING,
    SILICONFLOW_HTTP_RETRIES,
    SILICONFLOW_MAX_TOKENS,
    SILICONFLOW_MODEL,
    SILICONFLOW_OPENAI_BASE_URL,
    SILICONFLOW_TEMPERATURE,
    SILICONFLOW_TIMEOUT_SECONDS,
)
from data import build_comment_blocks, load_posts  # noqa: E402

print("PROJECT_ROOT =", PROJECT_ROOT)
print("model =", SILICONFLOW_MODEL)
if not SILICONFLOW_API_KEY:
    print("WARNING: SILICONFLOW_API_KEY 未设置；要真发请求需先在 .env 配置。")

In [ ]:
SYSTEM_PROMPT = """\
You are a financial-sentiment classifier for one short social-media discussion about a crypto asset.
Predict whether the asset price is more likely to go up (BULLISH) or down (BEARISH) over the time window in the user prompt.
Use only the post text, the root comment text, and the reply text provided. Do not use any future price, ground-truth label, or external data.
Output pure JSON only, no markdown, no prose:
{\"verdict\": \"BULLISH\" | \"BEARISH\", \"confidence\": 0.0, \"reason\": \"<short rationale grounded in supplied text>\"}
Rules: verdict must be exactly BULLISH or BEARISH; confidence in [0, 1]; if text is balanced, still pick one direction but lower confidence.
"""

USER_PROMPT_TEMPLATE = """\
Time window to forecast: {t_window}
Products mentioned: {products}
Post content: {post_content}
Root comment ({author}): {root_text}
Replies (up to 5):
{replies}
Decide BULLISH or BEARISH for the price in the next {t_window} after the root comment. Return JSON only.
"""

CACHE_DIR = Path(SILICONFLOW_CACHE_DIR).parent / "direct_llm"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _cache_path(payload):
    canonical = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    digest = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{digest}.json"


def _post_chat(payload):
    if not SILICONFLOW_API_KEY:
        raise RuntimeError("Missing SILICONFLOW_API_KEY. Set it in .env before calling the LLM.")
    request = urllib.request.Request(
        url=f"{SILICONFLOW_OPENAI_BASE_URL.rstrip('/')}/chat/completions",
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={
            "content-type": "application/json",
            "authorization": f"Bearer {SILICONFLOW_API_KEY}",
        },
        method="POST",
    )
    last_exc = None
    for attempt in range(1, SILICONFLOW_HTTP_RETRIES + 1):
        try:
            with urllib.request.urlopen(request, timeout=SILICONFLOW_TIMEOUT_SECONDS) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except (urllib.error.URLError, TimeoutError, socket.timeout, ssl.SSLError,
                ConnectionResetError, http.client.RemoteDisconnected, http.client.IncompleteRead) as exc:
            last_exc = exc
            if attempt == SILICONFLOW_HTTP_RETRIES:
                raise RuntimeError(f"LLM request failed after retries: {exc}") from exc
            time.sleep(0.8 * attempt)
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"LLM HTTP {exc.code}: {detail}") from exc
    raise RuntimeError(f"LLM request failed: {last_exc}")


def _extract_text(response):
    choices = response.get("choices") or []
    first = choices[0] if choices else {}
    message = first.get("message") or {}
    content = message.get("content")
    if isinstance(content, str):
        return content
    if isinstance(first.get("text"), str):
        return first["text"]
    raise ValueError("LLM response missing choices[0].message.content")


def _parse_json(text):
    raw = text.strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{[\s\S]*\}", raw)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass
    upper = raw.upper()
    return {"verdict": "BEARISH" if "BEARISH" in upper else "BULLISH", "confidence": 0.0, "reason": "fallback-parse"}


def classify_block(block):
    replies_blob = "\n--\n".join(
        f"- {r.author}: {(r.text or '').strip()[:300]}" for r in (block.replies or [])[:5]
    ) or "(no replies)"
    user_prompt = USER_PROMPT_TEMPLATE.format(
        t_window=block.t_window or "24h",
        products=",".join(block.products or []) or "(unknown)",
        post_content=(block.post_content or "").strip()[:2000],
        author=block.root_comment.author or "(unknown)",
        root_text=(block.root_comment.text or "").strip()[:2000],
        replies=replies_blob,
    )
    payload = {
        "model": SILICONFLOW_MODEL,
        "max_tokens": SILICONFLOW_MAX_TOKENS,
        "temperature": SILICONFLOW_TEMPERATURE,
        "enable_thinking": SILICONFLOW_ENABLE_THINKING,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    }
    cache_path = _cache_path(payload)
    if cache_path.exists():
        response = json.loads(cache_path.read_text(encoding="utf-8"))
    else:
        response = _post_chat(payload)
        try:
            cache_path.write_text(json.dumps(response, ensure_ascii=False, indent=2), encoding="utf-8")
        except OSError:
            pass
    text = _extract_text(response)
    obj = _parse_json(text)
    verdict = str(obj.get("verdict") or "").strip().upper()
    if verdict not in ("BULLISH", "BEARISH"):
        verdict = "BEARISH"
    try:
        confidence = float(obj.get("confidence", 0.0))
    except (TypeError, ValueError):
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))
    return verdict, confidence, str(obj.get("reason") or "").strip()[:300]

print("LLM client helpers defined. cache_dir =", CACHE_DIR)

In [ ]:
# 一行开关：True = 仅前 N 个块烟雾测试；False = 跑全量
SMOKE = True
LIMIT_BLOCKS = 5 if SMOKE else None

posts = load_posts(DEFAULT_INPUT_PATH)
blocks, issues = build_comment_blocks(posts)
if LIMIT_BLOCKS is not None:
    blocks = blocks[:LIMIT_BLOCKS]
print(f"blocks={len(blocks)} issues={len(issues)} smoke={SMOKE}")

LABEL_BULLISH, LABEL_BEARISH = 1, -1
predictions = []
for idx, block in enumerate(blocks, start=1):
    try:
        verdict, confidence, reason = classify_block(block)
    except Exception as exc:
        verdict, confidence, reason = "BEARISH", 0.0, f"error:{exc}"
    predictions.append({
        "block_id": block.block_id,
        "t0": block.t0.strftime("%Y-%m-%d %H:%M:%S"),
        "true_label": int(block.label),
        "verdict": verdict,
        "confidence": confidence,
        "reason": reason,
    })
    if idx % 5 == 0 or idx == len(blocks):
        print(f"  {idx}/{len(blocks)} done")

In [ ]:
def to_int(verdict):
    return LABEL_BULLISH if verdict == "BULLISH" else LABEL_BEARISH


pairs = [(p["true_label"], to_int(p["verdict"])) for p in predictions]
n = len(pairs)
correct = sum(1 for t, p in pairs if t == p)
acc = correct / n if n else 0.0
matrix = {"bullish": {"bullish": 0, "bearish": 0}, "bearish": {"bullish": 0, "bearish": 0}}
for t, p in pairs:
    matrix["bullish" if t == 1 else "bearish"]["bullish" if p == 1 else "bearish"] += 1
print(f"n={n}  acc={acc:.3f}")
print(f"confusion_matrix={matrix}")
print("--- 样例 ---")
for p in predictions:
    print(f"- {p['block_id']} | true={p['true_label']:+d} | verdict={p['verdict']} | conf={p['confidence']:.2f} | reason={p['reason'][:80]}")

OUT_DIR = PROJECT_ROOT / "outputs" / "direct_llm_smoke"
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "config": {
        "input": str(DEFAULT_INPUT_PATH),
        "limit_blocks": LIMIT_BLOCKS,
        "smoke": SMOKE,
        "model": SILICONFLOW_MODEL,
    },
    "total": n,
    "accuracy": acc,
    "confusion_matrix": matrix,
    "predictions": predictions,
}
(OUT_DIR / "metrics.json").write_text(
    json.dumps({k: v for k, v in summary.items() if k != "predictions"}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
with (OUT_DIR / "predictions.jsonl").open("w", encoding="utf-8") as fp:
    for row in predictions:
        fp.write(json.dumps(row, ensure_ascii=False) + "\n")
print("wrote", OUT_DIR / "metrics.json")
print("wrote", OUT_DIR / "predictions.jsonl")

## 如何切到全量

烟雾版跑通后，把第三个代码 cell 顶部的开关改成：

```python
SMOKE = False
LIMIT_BLOCKS = None  # 全量
```

再把输出目录改成 `outputs/direct_llm/`，整本 Run All。

服务器上无交互跑：

```bash
"D:/anaconda/envs/sentiment/python.exe" -m jupyter nbconvert --to notebook --execute \
    --ExecutePreprocessor.timeout=7200 \
    notebooks/03_direct_llm_baseline.ipynb \
    --output 03_direct_llm_baseline_full.ipynb
```

备注：

- 每个 block 一次 chat 调用；token 与时间成本都按 block 数线性增加。
- LLM 响应按 payload SHA256 缓存到 `outputs/llm_cache/direct_llm/`，重复运行不重复扣费。
- 烟雾版同样遵守 `CLAUDE.md` 硬性约束：prompt 里没有 `p1`、未来价格、真实标签。